# DENT-AI — Colab eğitimi (T4 GPU)
**Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU.**

Drive'da şu iki dosya olmalı: `MyDrive/DENT-AI/DENT-AI.zip` (proje) ve `MyDrive/DENT-AI/findings.zip` (hazırlanmış veri seti).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/DENT-AI/DENT-AI.zip -d /content
%cd /content/DENT-AI
!pip install -q ultralytics PyYAML opencv-python torchvision

## 1) Veri
Drive'dan yerel diske kopyalanıp açılır (Drive üzerinden doğrudan okumak çok yavaştır).

In [ ]:
!mkdir -p /content/datasets
!cp /content/drive/MyDrive/DENT-AI/findings.zip /content/datasets/
!unzip -q -o /content/datasets/findings.zip -d /content/datasets
!rm /content/datasets/findings.zip
!ls /content/datasets/findings && cat /content/datasets/findings/data.yaml

## 2) Eğitim — 640 px, 100 epoch, Early Stopping (patience=20)
Çıktılar Drive'a yazılır: oturum kapansa da kaybolmaz. Sonunda **test kümesi** skorları otomatik yazdırılır.

In [ ]:
SAVE = '/content/drive/MyDrive/DENT-AI'
!python -m dentai.train.train --task findings --data /content/datasets/findings/data.yaml --imgsz 640 --epochs 100 --patience 20 --project $SAVE/runs --save-dir $SAVE/models

### Oturum yarıda kesilirse
Yeniden bağlanın, 1. hücre ve veri hücresini çalıştırın, `SAVE = '/content/drive/MyDrive/DENT-AI'` satırını tekrar yazın, sonra eğitim hücresi yerine bunu çalıştırın:

In [ ]:
!python -m dentai.train.train --task findings --data /content/datasets/findings/data.yaml --project $SAVE/runs --save-dir $SAVE/models --resume $SAVE/runs/findings/weights/last.pt

## 3) Sonuç
`models/findings_best.pt`, `findings_best.json` ve `findings_test_metrics.json` dosyalarını bilgisayarınızdaki `DENT-AI/models/` içine koyun. Eğri grafikleri: `runs/findings/results.png`, `confusion_matrix.png`.

In [ ]:
!ls -la $SAVE/models
!cat $SAVE/models/findings_test_metrics.json

## 4) Diş numaralandırma (FDI) modeli — DENTEX
Drive'a `MyDrive/DENT-AI/teeth.zip` yükleyin (DENTEX'ten hazırlanmış: 634 röntgen, 32 diş, %70/15/15).
Önce 1. hücreyi, sonra aşağıdaki hücreyi çalıştırın. Dişler panoramikte küçük olduğu için 1024 px kullanılır.
Bitince `teeth_best.pt`, `teeth_best.json`, `teeth_test_metrics.json` dosyalarını bilgisayardaki `DENT-AI/models/` içine koyun.

In [ ]:
!mkdir -p /content/datasets
!cp /content/drive/MyDrive/DENT-AI/teeth.zip /content/datasets/
!unzip -q -o /content/datasets/teeth.zip -d /content/datasets
!rm /content/datasets/teeth.zip
!ls /content/datasets/teeth/images/* | head -3 ; cat /content/datasets/teeth/data.yaml | head -8
SAVE = '/content/drive/MyDrive/DENT-AI'
!python -m dentai.train.train --task teeth --data /content/datasets/teeth/data.yaml --imgsz 1024 --batch 8 --epochs 150 --patience 30 --project $SAVE/runs --save-dir $SAVE/models
!cat $SAVE/models/teeth_test_metrics.json

## 5) Bulgu modeli v2: sızıntısız %70/15/15, 1024 px
Drive'a `MyDrive/DENT-AI/findings_v2.zip` yükleyin. Eski `findings.zip`'in test kümesindeki görüntülerin %96'sı train'de de vardı (Roboflow kopyaları). v2'de bu sızıntı giderildi, bu yüzden skor eskisiyle doğrudan karşılaştırılamaz.
Çürük lezyonları çok küçük olduğu için 1024 px kullanılıyor. Eğitim uzun sürer. Oturum koparsa alttaki "devam" hücresini çalıştırın. Çıktılar `runs_v2` klasörüne gider, eski eğitim kayıtları korunur.

In [ ]:
!mkdir -p /content/datasets
!cp /content/drive/MyDrive/DENT-AI/findings_v2.zip /content/datasets/
!unzip -q -o /content/datasets/findings_v2.zip -d /content/datasets
!rm /content/datasets/findings_v2.zip
!cat /content/datasets/findings/data.yaml | tail -8
SAVE = '/content/drive/MyDrive/DENT-AI'
!python -m dentai.train.train --task findings --data /content/datasets/findings/data.yaml --imgsz 1024 --batch 16 --epochs 100 --patience 20 --project $SAVE/runs_v2 --save-dir $SAVE/models
!cat $SAVE/models/findings_test_metrics.json

In [ ]:
# DEVAM: oturum koptuysa 1. hücreden sonra yalnızca bunu çalıştırın
SAVE = '/content/drive/MyDrive/DENT-AI'
!mkdir -p /content/datasets
![ -d /content/datasets/findings/images ] || (cp /content/drive/MyDrive/DENT-AI/findings_v2.zip /content/datasets/ && unzip -q -o /content/datasets/findings_v2.zip -d /content/datasets && rm /content/datasets/findings_v2.zip)
!python -m dentai.train.train --task findings --data /content/datasets/findings/data.yaml --imgsz 1024 --project $SAVE/runs_v2 --save-dir $SAVE/models --resume $SAVE/runs_v2/findings/weights/last.pt
!cat $SAVE/models/findings_test_metrics.json